# 1 · What a data pipeline actually is

No experience needed. By the end of this notebook you will know what the job is
and what the words mean. In notebook 2 you write one.

---

## Start with a problem you already understand

KERB is a ride hailing company. When you book a ride, a row goes into a
PostgreSQL table called `kerb.trips`. That database has exactly one job: serve
riders and drivers, fast, right now.

Now the finance team asks: **how much did we earn last month?**

The obvious answer is *"query `kerb.trips`"*. Three things go wrong.

![](img/intro-1-why.png)

## So: make a copy

On our own hardware, on our own clock, shaped for the questions people actually
ask. **Everything in this course is that one idea, done carefully.**

In [ ]:
import sys; sys.path.insert(0, '.')
from nb import show, sql, fetch, run, counts

import psycopg
from pipelines.lib.config import dsn, SCHEMA

print('our schema is', SCHEMA)

---

## Look at the source before writing anything

Rule one of this job, and the one people skip.

In [ ]:
sql("""
    SELECT trip_id, requested_at, rider_id, driver_id, pu_zone_id,
           distance_km, duration_s, status
    FROM kerb.trips ORDER BY requested_at DESC LIMIT 5
""", 'kerb.trips, the newest five rides')

In [ ]:
sql("""
    SELECT count(*) AS rides,
           min(date(requested_at)) AS first_day,
           max(date(requested_at)) AS last_day
    FROM kerb.trips
""", 'how much is there')

---

## The words you will hear, defined once

| word | what it means |
|---|---|
| **source** | a system that already exists and is not yours. You are a guest |
| **warehouse** | your copy, on your clock, shaped for questions |
| **pipeline** | a program that moves a defined slice from one to the other |
| **batch** | the records one run of a pipeline is holding |
| **window** | the slice of time this run is responsible for |
| **contract** | the values you already know how to read |
| **quarantine** | where a record goes when it breaks the contract |
| **run log** | one row per run, saying what happened |
| **idempotent** | running it twice is the same as running it once |

## Bronze, silver, gold

![](img/intro-2-layers.png)

The names do not matter. **The discipline does.** Each layer has one job, and
you can always walk backwards: from a number in gold, to the row in silver, to
the row in bronze, to the source.

---

## The sources you will actually read

Four of them, all real, all running on this laptop right now. Each one has a
different way of being awkward, and that is the point.

In [ ]:
import subprocess

out = subprocess.run(['docker', 'ps', '--format', '{{.Names}}\t{{.Status}}'],
                     capture_output=True, text=True).stdout

for line in sorted(out.strip().splitlines()):
    name, _, status = line.partition('\t')
    if name.startswith('kerb-'):
        print(f'  {name:24} {status}')

| notebook | source | the awkward part |
|---|---|---|
| 2 | PostgreSQL `kerb.trips` | enormous, so you take a window |
| 3 | Kafka `kerb.trips.lifecycle` | your position lives on the broker, not in your database |
| 4 | MongoDB `driver_app_events` | no schema. Every release can change the shape |
| 5 | PayNimbus, over HTTP | a partner. Answers late, partially, or not at all |
| 6 | MinIO, gzipped CSV | no types at all. Everything is a string |

---

## What separates a pipeline from a script

![](img/intro-3-rules.png)

Those four rules are the entire course. Every pipeline you write obeys all four,
and `pipelines/lib/run.py` is where they are written down once so you do not
have to remember them.

## Here is one of them, right now

In [ ]:
sql(f"""
    SELECT table_name FROM information_schema.tables
    WHERE table_schema = '{SCHEMA}' ORDER BY table_name
""", f"what is in '{SCHEMA}' before you start")

In [ ]:
run('cli.py', 'status')

---

## Where everything lives

| | |
|---|---|
| `pipelines/` | the eight pipelines, one file each |
| `pipelines/lib/` | the four rules, written once |
| `signals/` | the board that notices when a number moves |
| `notebooks/` | these lessons |
| `cli.py` | `status`, `run all`, `board`, `reset`, `break`, `fix` |

And one command matters more than the rest:

```bash
python cli.py reset
```

**Everything in this course can be put back to nothing and rebuilt in seconds.**
Nothing you do can leave the estate in a state you cannot get out of, which is
what makes it safe to break things on purpose in notebook 9.

## What you learned

- A warehouse exists so reporting does not compete with the product
- **Look at the source first.** Shape, size, date range
- **Bronze copies. Silver joins. Gold answers.**
- A pipeline is idempotent, atomic, honest and observable. A script is none of those
- Four sources, four different ways of being awkward
- There is always a way back to zero